<a href="https://colab.research.google.com/github/lukaszd96/machine-learning-course/blob/main/unsupervised/03_association_rules/02_apriori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### scikit-learn
Strona biblioteki: [https://scikit-learn.org](https://scikit-learn.org)  

Dokumentacja/User Guide: [https://scikit-learn.org/stable/user_guide.html](https://scikit-learn.org/stable/user_guide.html)

Podstawowa biblioteka do uczenia maszynowego w języku Python.

Aby zainstalować bibliotekę scikit-learn, użyj polecenia poniżej:
```
!pip install scikit-learn
```
Aby zaktualizować do najnowszej wersji bibliotekę scikit-learn, użyj polecenia poniżej:
```
!pip install --upgrade scikit-learn
```
Kurs stworzony w oparciu o wersję `0.22.1`

### Spis treści:
1. [Import bibliotek](#0)
2. [Załadownaie danych](#1)
3. [Przygotowanie danych](#2)
4. [Kodowanie transakcji](#3)
5. [Algorytm Apriori](#4)




In [30]:
import warnings
warnings.simplefilter("ignore", DeprecationWarning)

import os
os.environ["PYTHONWARNINGS"] = "ignore::DeprecationWarning"

### <a name='0'></a> Import bibliotek

In [31]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

pd.set_option('display.float_format', lambda x: f'{x:.2f}')

### <a name='1'></a> Załadownaie danych

In [32]:
!wget https://storage.googleapis.com/esmartdata-courses-files/ml-course/products.csv
!wget https://storage.googleapis.com/esmartdata-courses-files/ml-course/orders.csv

--2026-02-20 13:31:39--  https://storage.googleapis.com/esmartdata-courses-files/ml-course/products.csv
Resolving storage.googleapis.com (storage.googleapis.com)... 173.194.69.207, 173.194.79.207, 108.177.96.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|173.194.69.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2166953 (2.1M) [application/octet-stream]
Saving to: ‘products.csv.2’

products.csv.2      100%[===================>]   2.07M  --.-KB/s    in 0.1s    

2026-02-20 13:31:40 (20.6 MB/s) - ‘products.csv.2’ saved [2166953/2166953]

--2026-02-20 13:31:40--  https://storage.googleapis.com/esmartdata-courses-files/ml-course/orders.csv
Resolving storage.googleapis.com (storage.googleapis.com)... 173.194.69.207, 173.194.79.207, 108.177.96.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|173.194.69.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 24680147 (24M) [application/octet-stre

In [33]:
products = pd.read_csv('products.csv', usecols=['product_id', 'product_name'])
products.head()

,product_id,product_name
0,1,Chocolate Sandwich Cookies
1,2,All-Seasons Salt
2,3,Robust Golden Unsweetened Oolong Tea
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...
4,5,Green Chile Anytime Sauce


In [34]:
orders = pd.read_csv('orders.csv', usecols=['order_id', 'product_id'])
orders.head()

,order_id,product_id
0,1,49302
1,1,11109
2,1,10246
3,1,49683
4,1,43633


### <a name='2'></a> Przygotowanie danych

In [35]:
data = pd.merge(orders, products, how='inner', on='product_id', sort=True)
data = data.sort_values(by='order_id')
data.head()

,order_id,product_id,product_name
277441,1,11109,Organic 4% Milk Fat Whole Milk Cottage Cheese
1194893,1,43633,Lightly Smoked Sardines in Olive Oil
256931,1,10246,Organic Celery Hearts
588447,1,22035,Organic Whole String Cheese
1302365,1,47209,Organic Hass Avocado


In [36]:
data.describe()

,order_id,product_id
count,1384617.00,1384617.00
mean,1706297.62,25556.24
std,989732.65,14121.27
min,1.00,1.00
25%,843370.00,13380.00
50%,1701880.00,25298.00
75%,2568023.00,37940.00
max,3421070.00,49688.00


In [37]:
# rozkład produktów
data['product_name'].value_counts()

,count
product_name,
Banana,18726
Bag of Organic Bananas,15480
Organic Strawberries,10894
Organic Baby Spinach,9784
Large Lemon,8135
...,...
Lemon Butter Cuticle Cream,1
Cerveza Especial Imported Beer From Mexico,1
High Fiber Bran Crispbread,1


In [38]:
# liczba transakcji
data['order_id'].nunique()

131209

In [39]:
transactions = data.groupby(by='order_id')['product_name'].apply(lambda name: ','.join(name))
transactions

,product_name
order_id,
1,"Organic 4% Milk Fat Whole Milk Cottage Cheese,..."
36,"Organic Garnet Sweet Potato (Yam),Asparagus,Sp..."
38,"Organic Raw Unfiltered Apple Cider Vinegar,Org..."
96,"Roasted Turkey,Organic Grape Tomatoes,Organic ..."
98,"Organic Raw Kombucha Gingerade,Organic 2% Butt..."
...,...
3421049,"Organic Baby Kale Mix,Lemon Sports Drink,Organ..."
3421056,"Homestyle Classics Meatloaf,Tartar Sauce,Total..."
3421058,Wine Infused Salame Cheese and Crackers Small ...


In [40]:
transactions = transactions.str.split(',')
transactions

,product_name
order_id,
1,[Organic 4% Milk Fat Whole Milk Cottage Cheese...
36,"[Organic Garnet Sweet Potato (Yam), Asparagus,..."
38,"[Organic Raw Unfiltered Apple Cider Vinegar, O..."
96,"[Roasted Turkey, Organic Grape Tomatoes, Organ..."
98,"[Organic Raw Kombucha Gingerade, Organic 2% Bu..."
...,...
3421049,"[Organic Baby Kale Mix, Lemon Sports Drink, Or..."
3421056,"[Homestyle Classics Meatloaf, Tartar Sauce, To..."
3421058,[Wine Infused Salame Cheese and Crackers Small...


### <a name='3'></a> Kodowanie transakcji

In [41]:
from mlxtend.preprocessing import TransactionEncoder

encoder = TransactionEncoder()
encoder.fit(transactions)
transactions_encoded = encoder.transform(transactions, sparse=True)
transactions_encoded

<Compressed Sparse Row sparse matrix of dtype 'bool'
	with 1442410 stored elements and shape (131209, 40434)>

In [42]:
transactions_encoded_df = pd.DataFrame(transactions_encoded.toarray(), columns=encoder.columns_)
transactions_encoded_df

,,Apricot & Banana Stage 2 Baby Food,Broad Spectrum SPF 30,Instant,Livermore Valley,Low Sodium Marinara,Premium,Vetiver scent,Whole,#2,...,with Xylitol Cinnamon 18 Sticks Sugar Free Gum,with Xylitol Island Berry Lime 18 Sticks Sugar Free Gum,with Xylitol Minty Sweet Twist 18 Sticks Sugar Free Gum,with Xylitol Original Flavor 18 Sticks Sugar Free Gum,with Xylitol Unwrapped Original Flavor 50 Sticks Sugar Free Gum,with Xylitol Unwrapped Spearmint 50 Sticks Sugar Free Gum,with Xylitol Watermelon Twist 18 Sticks Sugar Free Gum,with a Splash of Mango Coconut Water,with a Splash of Pineapple Coconut Water,Lightly Seasoned with Rosemary and Roasted Garlic Family Size Herb Chicken Tortellini
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131204,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
131205,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
131206,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
131207,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


### <a name='4'></a> Algorytm Apriori

In [43]:
from mlxtend.frequent_patterns import apriori, association_rules

supports = apriori(transactions_encoded_df, min_support=0.01, use_colnames=True)
supports = supports.sort_values(by='support', ascending=False)
supports.head(10)

,support,itemsets
8,0.14,(Banana)
7,0.12,(Bag of Organic Bananas)
76,0.08,(Organic Strawberries)
41,0.07,(Organic Baby Spinach)
31,0.06,(Large Lemon)
37,0.06,(Organic Avocado)
61,0.06,(Organic Hass Avocado)
100,0.05,(Strawberries)
33,0.05,(Limes)
69,0.04,(Organic Raspberries)


In [44]:
rules = association_rules(supports, metric='confidence', min_threshold=0)
rules = rules.iloc[:, [0, 1, 4, 5, 6]]
rules = rules.sort_values(by='lift', ascending=False)
rules.head(15)

,antecedents,consequents,support,confidence,lift
27,( Bag),(Clementines),0.01,0.79,36.84
26,(Clementines),( Bag),0.01,0.52,36.84
22,(Limes),(Large Lemon),0.01,0.26,4.26
23,(Large Lemon),(Limes),0.01,0.20,4.26
19,(Organic Raspberries),(Organic Strawberries),0.01,0.30,3.63
18,(Organic Strawberries),(Organic Raspberries),0.01,0.15,3.63
31,(Organic Avocado),(Large Lemon),0.01,0.18,2.94
30,(Large Lemon),(Organic Avocado),0.01,0.17,2.94
2,(Organic Hass Avocado),(Bag of Organic Bananas),0.02,0.33,2.81
3,(Bag of Organic Bananas),(Organic Hass Avocado),0.02,0.16,2.81
